In [ ]:
import pandas as pd
import requests
from io import StringIO
from functools import reduce
import time 

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# date	
# month	
# employment_rate_diff	
# industry_production_prices_diff	
# companies_trust	
# rolling_mean_6m	
# rolling_mean_12m	
# PCA_labour_market_1	
# PCA_labour_market_2	
# PCA_price_energy_1	
# PCA_price_energy_2	
# unemployment_rate_diff 
# -------------------------------

# 'gasoline_car_price, 
# 'diesel_price', 
# 'GPL_price', 
# 'diesel_heatining_price'

# 'workforce', 
# 'inactives', 
# 'activity_rate'

# 'employment_rate'
# 'industry_production_prices'
# 'companies_trust'

# 'unemployment_rate'

In [46]:
def API_calling(dataflow, start_date = '2013-01', end_date = '2026-07'):
    # API endpoint and dataset
    base = "https://esploradati.istat.it/SDMXWS/rest/data"
    # dataflow = "IT1,150_875"
    dataflow = dataflow

    # Query parameters
    params = {
        "startPeriod": f"{start_date}",
        "endPeriod": f"{end_date}",
        #"POSIZ_PROF": "1", #Dipedenti
        #"PERM_TEMP_EMPLOYEES": ""
        "format": "CSV"  # request CSV
    }

    # Headers (fixed syntax)
    headers = {"Accept": "text/csv"}

    # Build URL
    url = f"{base}/{dataflow}"

    # Send GET request
    response = requests.get(url, params=params, headers=headers, verify=False)

    # Check for errors
    if response.status_code != 200:
        raise Exception(f"Error {response.status_code}: {response.text}")

    # Read CSV into pandas directly
    csv_data = StringIO(response.text)
    df = pd.read_csv(csv_data)

    return df
    

In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,150_875_DF_DCCV_OCCUPATIMENS1_1")

#"POSIZ_PROF"--> #'posizione professionale': # 1: Dipedenti, # 2: Indipendenti,# 9: Totale
#"PERM_TEMP_EMPLOYEES" --> 'Categoria occupazione dipendente' #1: Tempo determinato, 2: Tempo indeterminato,9: Totale

last_edition = df['EDITION'].unique().max()
df = df[df['EDITION'] == last_edition]

df = df[(df['ADJUSTMENT'] == 'N') #prendo  solo le righe che NON hanno adjustments
      & (df['POSIZ_PROF']==1)  # prendo solo i lavoratori Dipendenti
      & (df['PERM_TEMP_EMPLOYEES']!=9)] # prendo solo il valore totale

df['PERM_TEMP_EMPLOYEES'].replace({1:'fixed_term', 2 : 'permanent'}, inplace = True)
df_employment_pivoted = df.pivot(index='TIME_PERIOD', columns='PERM_TEMP_EMPLOYEES', values=['OBS_VALUE'])

df_employment_pivoted = df_employment_pivoted.rename_axis(None)

df_employment_pivoted.columns = df_employment_pivoted.columns.droplevel(0)

df_employment_pivoted['date'] =  pd.to_datetime(df_employment_pivoted.index, format='%Y-%m')

df_employment_pivoted = df_employment_pivoted.rename_axis(None, axis=1)

df_employment_pivoted = df_employment_pivoted.reset_index()#
df_employment_pivoted.drop('index', axis = 1, inplace = True)#

In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,150_872_DF_DCCV_TAXOCCUMENS1_1")

last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_employed_rate = df[(df.SEX==9) & (df.AGE.isin(['Y15-64','Y15-34'])) & (df.ADJUSTMENT == 'N')] # I select youth employment and full age range employment

df_employed_rate_pivoted = df_employed_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_employed_rate_pivoted = df_employed_rate_pivoted.rename_axis(None)

df_employed_rate_pivoted.columns = df_employed_rate_pivoted.columns.droplevel(0)

df_employed_rate_pivoted['date'] =  pd.to_datetime(df_employed_rate_pivoted.index, format='%Y-%m')

df_employed_rate_pivoted = df_employed_rate_pivoted.rename_axis(None, axis=1)

df_employed_rate_pivoted.rename(columns={'Y15-64': 'employment rate (Y15-64)', 'Y15-34':  'youth employment rate (Y15-34)'}, inplace = True)

df_employed_rate_pivoted = df_employed_rate_pivoted.reset_index()#
df_employed_rate_pivoted.drop('index', axis = 1, inplace = True)#


In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,150_873_DF_DCCV_FORZLVMENS1_1")

last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_workforce = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_workforce_pivoted = df_workforce.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_workforce_pivoted = df_workforce_pivoted.rename_axis(None)

df_workforce_pivoted.columns = df_workforce_pivoted.columns.droplevel(0)

df_workforce_pivoted['date'] =  pd.to_datetime(df_workforce_pivoted.index, format='%Y-%m')

df_workforce_pivoted = df_workforce_pivoted.rename_axis(None, axis=1)

df_workforce_pivoted.rename(columns={'Y15-64': 'workforce (Y15-64)'}, inplace = True)

df_workforce_pivoted = df_workforce_pivoted.reset_index()#
df_workforce_pivoted.drop('index', axis = 1, inplace = True)#


In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,150_876_DF_DCCV_TAXATVTMENS1_1")

last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_activity_rate = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_activity_rate_pivoted = df_activity_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_activity_rate_pivoted = df_activity_rate_pivoted.rename_axis(None)

df_activity_rate_pivoted.columns = df_activity_rate_pivoted.columns.droplevel(0)

df_activity_rate_pivoted['date'] =  pd.to_datetime(df_activity_rate_pivoted.index, format='%Y-%m')

df_activity_rate_pivoted = df_activity_rate_pivoted.rename_axis(None, axis=1)

df_activity_rate_pivoted.rename(columns={'Y15-64': 'activity rate (Y15-64)'}, inplace =True)

df_activity_rate_pivoted = df_activity_rate_pivoted.reset_index()#
df_activity_rate_pivoted.drop('index', axis = 1, inplace = True)#

In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,151_874_DF_DCCV_TAXDISOCCUMENS1_1")

last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_unemployment_rate = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_unemployment_rate_pivoted = df_unemployment_rate.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.rename_axis(None)

df_unemployment_rate_pivoted.columns = df_unemployment_rate_pivoted.columns.droplevel(0)

df_unemployment_rate_pivoted['date'] =  pd.to_datetime(df_unemployment_rate_pivoted.index, format='%Y-%m')#

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.rename_axis(None, axis=1)

df_unemployment_rate_pivoted.rename(columns={'Y15-64': 'unemployment rate (Y15-64)'}, inplace = True)

df_unemployment_rate_pivoted = df_unemployment_rate_pivoted.reset_index()#

df_unemployment_rate_pivoted.drop('index', axis = 1, inplace = True)#

In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,152_879_DF_DCCV_INATTIVMENS1_1")

last_edition = df['EDITION'].unique().max() # this is necessary to take the most updated values
df = df[df['EDITION'] == last_edition]

df_inactives = df[(df.SEX==9) & (df.AGE=='Y15-64') & (df.ADJUSTMENT == 'N')]

df_inactives_pivoted = df_inactives.pivot(index='TIME_PERIOD', columns='AGE', values=['OBS_VALUE'])

df_inactives_pivoted = df_inactives_pivoted.rename_axis(None)

df_inactives_pivoted.columns = df_inactives_pivoted.columns.droplevel(0)

df_inactives_pivoted['date'] =  pd.to_datetime(df_inactives_pivoted.index, format='%Y-%m')#

df_inactives_pivoted = df_inactives_pivoted.rename_axis(None, axis=1)

df_inactives_pivoted.rename(columns={'Y15-64': 'inactives (Y15-64)'}, inplace = True)

df_inactives_pivoted = df_inactives_pivoted.reset_index()#

df_inactives_pivoted.drop('index', axis = 1, inplace = True)#


In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,145_360_DF_DCSC_PREZZPIND_1_4")

df_industry_production_prices = df[(df['MARKET']=='T')  & (df['ECON_ACTIVITY_NACE_2007']=='0020')] [['TIME_PERIOD', 'OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'industry_production_prices'})
df_industry_production_prices['date'] =  pd.to_datetime(df_industry_production_prices['date'], format='%Y-%m')#
df_industry_production_prices.drop('index', axis=1, inplace= True)

In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,145_376_DF_DCSC_PREZPRODSERV_1_7")

df_services_production_prices = df[(df['ECON_ACTIVITY_NACE_2007']=='H_N_XK')][['TIME_PERIOD','OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'services_production_prices'})
df_services_production_prices['date'] = df_services_production_prices['date'].apply(lambda x: pd.Period(x, freq="Q").to_timestamp(how="start")).dt.strftime("%Y-%m-%d")
df_services_production_prices['date'] =  pd.to_datetime(df_services_production_prices['date'], format='%Y-%m-%d')
df_services_production_prices.drop('index', axis=1, inplace= True)



In [ ]:
df = API_calling(start_date = '2023-01', end_date = '2026-07', dataflow = "IT1,6_64_DF_DCSC_IESI_2")

df_companies_trust = df[['TIME_PERIOD','OBS_VALUE']].reset_index().rename(columns={'TIME_PERIOD':'date', 'OBS_VALUE':'companies_trust'})
df_companies_trust['date'] =  pd.to_datetime(df_companies_trust['date'], format='%Y-%m')
df_companies_trust.drop('index', axis=1, inplace= True)

In [11]:
def get_fuel_prices(fuel_type: str, start_year: int):

    fuel_type_dict = {'gasoline': 1,
                        'diesel':   2,
                        'GPL':   5,
                        'diesel_heatining':  3}

    fuel_code = fuel_type_dict.get(fuel_type)    

    df = pd.read_csv(f'https://sisen.mase.gov.it/dgsaie/api/v1/monthly-prices/{fuel_code}/export?format=CSV&lang=it' )
    df["date"] = pd.to_datetime(df["ANNO"].astype(str) + "-" + df["CODICE_MESE"].astype(str) + "-01",  format="%Y-%m-%d")
    df[f"{fuel_type}_price"] = df["PREZZO"].str.replace(",", "", regex=False)
    df[f"{fuel_type}_price"] = df[f"{fuel_type}_price"].astype('float')
    df = df[df['ANNO']>=start_year][['date', f'{fuel_type}_price']].reset_index()
    df.drop('index', axis=1, inplace=True)

    return df


In [25]:
df_gasoline_car_price = get_fuel_prices(fuel_type = "gasoline", start_year = 2024)
df_diesel_price = get_fuel_prices(fuel_type = "diesel_heatining", start_year = 2024)
df_GPL_price  = get_fuel_prices(fuel_type = "GPL", start_year = 2024)
df_diesel_heatining_price = get_fuel_prices(fuel_type = "diesel_heatining", start_year = 2024)

In [ ]:
dfs = [df_employment_pivoted,
        df_employed_rate_pivoted,
        df_workforce_pivoted,
        df_activity_rate_pivoted,
        df_unemployment_rate_pivoted,
        df_inactives_pivoted,
        df_industry_production_prices,
        df_services_production_prices, 
        df_gasoline_car_price,
        df_diesel_price,
        df_GPL_price,
        df_diesel_heatining_price,
        df_companies_trust]

df1 = pd.merge(df_employment_pivoted, df_employed_rate_pivoted, on='date', how='outer')

df2 = pd.merge(df1, df_workforce_pivoted, on='date', how='outer')

df3 = pd.merge(df2, df_activity_rate_pivoted, on='date', how='outer')

df4 = pd.merge(df3, df_unemployment_rate_pivoted, on='date', how='outer')

df5 = pd.merge(df4, df_inactives_pivoted, on='date', how='outer')

df6 = pd.merge(df5, df_industry_production_prices, on='date', how='outer')

df7 = pd.merge(df6, df_services_production_prices, on='date', how='outer')

df8 = pd.merge(df7, df_gasoline_car_price, on='date', how='inner')

df9 = pd.merge(df8, df_diesel_price, on='date', how='inner')

df10 = pd.merge(df9, df_GPL_price, on='date', how='inner')

df11 = pd.merge(df10, df_diesel_heatining_price, on='date', how='inner')

final_df = pd.merge(df11, df_companies_trust, on='date', how='inner')

final_df

In [ ]:
final_df.rename(columns = {'workforce (Y15-64)': 'workforce', 
                           'inactives (Y15-64)': 'inactives',
                            'youth employment rate (Y15-34)': 'youth_employment_rate', 
                            # 'employment rate (Y15-34)': 'employment_rate', 
                            'employment rate (Y15-64)':'employment_rate', 
                            # 'youth employment rate (Y15-64)':'youth_employment_rate', 
                            'unemployment rate (Y15-64)': 'unemployment_rate', 
                            'activity rate (Y15-64)': 'activity_rate' }, inplace =True)


final_df = final_df[['date', #'fixed_term', 'permanent', 
                     'workforce', 'inactives',
                    'employment_rate', 'youth_employment_rate', 'activity_rate', 'unemployment_rate', 
                    'industry_production_prices', 'services_production_prices', 'companies_trust', 
                    'gasoline_car_price', 'diesel_price', 'GPL_price', 'diesel_heatining_price']]

In [ ]:
final_df[['employment_rate', 'youth_employment_rate', 
          'activity_rate', 'unemployment_rate']] = final_df[['employment_rate',   'activity_rate', 'unemployment_rate']].round(2)


In [2]:
df_merged = pd.read_csv("dataset_forecast.csv")
df_merged["date"] = pd.to_datetime(df_merged["date"], format='ISO8601' )
df_merged = df_merged[df_merged['date']<='2026-01-01'] # I filter only where I have rows without null values

In [3]:
meadian_comp_trust= df_merged['companies_trust'].median()
meadian_ind_price = df_merged['industry_production_prices'].median()
df_merged['companies_trust'] = df_merged['companies_trust'].fillna(meadian_comp_trust )
df_merged['industry_production_prices'] = df_merged['industry_production_prices'].fillna(meadian_ind_price )

In [ ]:
df_merged = df_merged[['date', 
                         'workforce', 
                         'inactives', 
                         'employment_rate',
                        # 'youth_employment_rate', 
                        'activity_rate', 
                        'unemployment_rate',
                        'industry_production_prices', 
                        'companies_trust', 
                        'gasoline_car_price',
                        'diesel_price', 
                        'GPL_price', 
                        'diesel_heatining_price']]

In [6]:
df_merged[[  'workforce', 'inactives']] = df_merged[[ 'workforce', 'inactives']]*1000

df_merged[[  'workforce', 'inactives']] = df_merged[[  'workforce', 'inactives']].astype(int)

In [29]:
non_stationary_cols = [
                'workforce',
                'inactives',
                'employment_rate',
                # 'youth_employment_rate',
                'activity_rate',
                'industry_production_prices',
                'gasoline_car_price',
                'diesel_price',
                'diesel_heatining_price',
                'unemployment_rate'
                ]


In [30]:
for col in non_stationary_cols:
    print(col)
    df_merged[f'{col}_diff'] = df_merged[col] - df_merged[col].shift( 1) # qui facciamo subito la differnziazione

workforce
inactives
employment_rate
activity_rate
industry_production_prices
gasoline_car_price
diesel_price
diesel_heatining_price
unemployment_rate


In [22]:
df_merged

,date,workforce,inactives,employment_rate,youth_employment_rate,activity_rate,unemployment_rate,industry_production_prices,companies_trust,gasoline_car_price,diesel_price,GPL_price,diesel_heatining_price
0,2013-01-01,24545012,14461505,39.96,54.54,62.93,13.32,94.2,77.7,1.74994,1.69438,0.87203,1.43948
1,2013-02-01,24651944,14350054,41.10,54.82,63.21,13.27,94.3,76.5,1.78100,1.69967,0.85676,1.45680
2,2013-03-01,24585629,14412556,40.20,54.94,63.04,12.86,94.3,78.1,1.79691,1.69394,0.84149,1.43179
3,2013-04-01,24489735,14505159,39.02,54.78,62.80,12.78,93.9,74.2,1.75380,1.65111,0.81166,1.40430
4,2013-05-01,24531196,14459758,40.39,55.04,62.92,12.51,93.9,75.9,1.71616,1.61231,0.77374,1.37599
...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,2025-09-01,24732640,12416623,44.11,62.55,66.58,6.04,123.1,93.8,1.71024,1.63531,0.69714,1.38680
153,2025-10-01,24712759,12433050,43.28,62.78,66.53,5.64,122.8,94.4,1.69451,1.62232,0.69545,1.38145
154,2025-11-01,24426135,12718409,43.00,62.13,65.76,5.52,124.0,96.1,1.71901,1.68175,0.69413,1.41570
155,2025-12-01,24531682,12609532,43.43,62.20,66.05,5.83,123.1,96.5,1.69686,1.65260,0.69482,1.35576


In [ ]:
df_merged_detrended =  df_merged[['date',   
                                    'workforce_diff', 
                                    'inactives_diff',  
                                    'employment_rate_diff', 
                                    'activity_rate_diff', 
                                    'industry_production_prices_diff',
                                    'companies_trust',  
                                    'gasoline_car_price_diff', 
                                    'diesel_price_diff',  
                                    'GPL_price',  
                                    'diesel_heatining_price_diff', 
                                    'unemployment_rate', 
                                    'unemployment_rate_diff'
                                    ]]

In [32]:
df_merged_detrended['rolling_mean_6m'] = df_merged_detrended['unemployment_rate_diff'].rolling(window=6).mean()
df_merged_detrended['rolling_mean_12m'] = df_merged_detrended['unemployment_rate_diff'].rolling(window=12).mean()

In [ ]:
df_merged_detrended = df_merged_detrended.iloc[13:] # questo serve a togliere le prime 13 righe che hanno dei valori NUll

X_labour_market = df_merged_detrended[[  'workforce_diff', 'inactives_diff', 'activity_rate_diff']]

X_price_energy = df_merged_detrended[[ 'gasoline_car_price_diff', 'diesel_price_diff', 'GPL_price', 'diesel_heatining_price_diff']]


scaler = StandardScaler()
X_labour_market_scaled = scaler.fit_transform(X_labour_market)
X_price_energy_scaled = scaler.fit_transform(X_price_energy)


pca = PCA(n_components=2)
df_merged_detrended[['PCA_labour_market_1', 'PCA_labour_market_2']] = pca.fit_transform(X_labour_market_scaled)
df_merged_detrended[['PCA_price_energy_1', 'PCA_price_energy_2']] = pca.fit_transform(X_price_energy)